# 🛡️ Community Shield (ประชาอารักษ์) — Fine-Tuning LoRA AI on Google Colab
### โครงการระบบบริหารจัดการสาธารณภัยชุมชนวัดเทวราชกุญชร เขตดุสิต กรุงเทพมหานคร

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/XevilA/thai-community-shield/blob/main/notebooks/community_shield_lora_training.ipynb)

สมุดบันทึกนี้ใช้สำหรับ **Fine-Tuning โมเดลภาษาขนาดใหญ่ (LLM)** ด้วยเทคนิค **LoRA (Low-Rank Adaptation, PEFT)** บน Google Colab (รองรับ GPU ฟรี T4 หรือ A100)
- **Base Model:** `Qwen/Qwen2.5-1.5B-Instruct` (4-bit QLoRA)
- **Dataset:** 1,200 สถานการณ์ภัยพิบัติน้ำท่วมริมแม่น้ำเจ้าพระยา (โทรมาตร LoRaWAN AS923, ยายสมจิตร A-012, สะพานไม้จมน้ำ)
- **Output:** `adapter_model.safetensors` ขนาดกะทัดรัด พร้อมนำไปรัน Offline 100% บนเครื่องระบบบัญชาการ

## 1. ตรวจสอบ GPU และติดตั้งไลบรารีที่จำเป็น

In [ ]:
# 1. เช็กความพร้อมของการ์ดจอ NVIDIA ใน Colab
!nvidia-smi

# 2. ติดตั้ง PyTorch + Torchvision ที่เข้ากันได้ (CUDA 12.1) และชุดไลบรารี Hugging Face
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q --upgrade transformers peft accelerate bitsandbytes datasets trl safetensors huggingface_hub

## 2. ดาวน์โหลดและเตรียมชุดข้อมูลภัยพิบัติชุมชน (Chao Phraya Flood Dataset)

In [ ]:
import os
import json
import urllib.request
from datasets import load_dataset

os.makedirs("data", exist_ok=True)

# ดาวน์โหลด Dataset จาก GitHub Repository ของโครงการ
train_url = "https://raw.githubusercontent.com/XevilA/thai-community-shield/main/ml/data/train.jsonl"
val_url = "https://raw.githubusercontent.com/XevilA/thai-community-shield/main/ml/data/val.jsonl"

print("📥 กำลังดาวน์โหลดชุดข้อมูล...")
urllib.request.urlretrieve(train_url, "data/train.jsonl")
urllib.request.urlretrieve(val_url, "data/val.jsonl")

dataset = load_dataset("json", data_files={"train": "data/train.jsonl", "validation": "data/val.jsonl"})
print("✅ โหลด Dataset สำเร็จ:", dataset)
print("\n--- ตัวอย่างข้อมูล (Sample 1) ---")
print(dataset["train"][0]["prompt"][:300] + "...")

## 3. โหลด Base Model และ Tokenizer (QLoRA 4-bit)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# แนะนำ Qwen2.5-1.5B-Instruct สำหรับ Colab Free T4 (รวดเร็ว ประหยัด VRAM และแม่นยำสูง)
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"⏳ กำลังโหลด Base Model: {MODEL_ID}...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

print("✅ โหลด Base Model สำเร็จเรียบร้อยบน GPU!")

## 4. กำหนดค่า LoRA Configuration (PEFT)

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

# ตั้งค่า LoRA Hyperparameters (Rank 16, Alpha 32)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

## 5. เริ่มต้นกระบวนการฝึกสอน (Fine-Tuning ด้วย SFTTrainer)

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

def formatting_prompts_func(batch):
    output_texts = []
    for p, c in zip(batch["prompt"], batch["completion"]):
        output_texts.append(f"{p}\n<response>\n{c}\n</response>")
    return output_texts

training_args = TrainingArguments(
    output_dir="./lora_checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    fp16=True,
    report_to="none"
)

trainer = SFTTrainer(
    model=peft_model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    formatting_func=formatting_prompts_func,
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args
)

print("🚀 เริ่มการฝึกสอน LoRA Model บน GPU Colab...")
trainer.train()
print("🎉 การฝึกสอนเสร็จสมบูรณ์ 100%!")

## 6. ทดสอบการวิเคราะห์ภัยพิบัติและ Chain-of-Thought ด้วย LoRA Adapter

In [ ]:
test_prompt = (
    "<telemetry>\n"
    "Node: GW-THEWARAT-BELF-01 | Freq: 923.4 MHz (AS923-TH)\n"
    "Water Level: +0.82 m MSL | Rate of Rise: +0.38 m/h\n"
    "RF Link: RSSI -84 dBm | SNR +10.5 dB | Battery 3.60V\n"
    "Target Vulnerable Household: A-012 (ยายสมจิตร รัตนประสิทธิ์, bedridden)\n"
    "Special Requirements: ถังออกซิเจนและเปลสนาม\n"
    "Oxygen Supply: 35 min remaining\n"
    "Current Boardwalk State: จมน้ำแล้ว (+20 ซม.)\n"
    "</telemetry>\n\n"
    "<instruction>\n"
    "วิเคราะห์ข้อมูลโทรมาตร LoRaWAN และสถานการณ์กลุ่มเปราะบางเพื่อประเมินความเสี่ยงและสั่งการอพยพเร่งด่วนตามกรอบ Chain-of-Thought\n"
    "</instruction>\n<response>\n"
)

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = trainer.model.generate(
    **inputs,
    max_new_tokens=300,
    temperature=0.2,
    repetition_penalty=1.1,
    do_sample=True
)

print("🤖 ผลการอนุมานของ LoRA Model:")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## 7. บันทึกและดาวน์โหลดไฟล์ Safetensors เพื่อนำไปใส่ในระบบจริง

In [ ]:
import shutil
from google.colab import files

OUTPUT_ADAPTER_DIR = "./thewarat-chao-phraya-disaster-lora"
trainer.model.save_pretrained(OUTPUT_ADAPTER_DIR, safe_serialization=True)
tokenizer.save_pretrained(OUTPUT_ADAPTER_DIR)

print(f"✅ บันทึกไฟล์น้ำหนัก Safetensors เรียบร้อยที่ {OUTPUT_ADAPTER_DIR}")
!ls -lh ./thewarat-chao-phraya-disaster-lora

# สร้างไฟล์ Zip สำหรับดาวน์โหลดไปวางในเครื่องแม่ข่าย (/Volumes/MAC/Thai_Community/ml/models/lora_adapter)
shutil.make_archive("thewarat_lora_adapter", "zip", OUTPUT_ADAPTER_DIR)
print("📦 กำลังดาวน์โหลด thewarat_lora_adapter.zip...")
files.download("thewarat_lora_adapter.zip")